<a href="https://colab.research.google.com/github/varakalicheva-hub/diplom_2026/blob/main/%D0%9A%D0%BE%D0%B4_%D0%B4%D0%BB%D1%8F_%D0%BE%D1%86%D0%B5%D0%BD%D0%BA%D0%B8_%D0%BA%D0%B0%D1%87%D0%B5%D1%81%D1%82%D0%B2%D0%BE_%D0%BF%D0%BE_%D1%81%D0%B5%D0%BC%D0%B0%D0%BD%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%BE%D0%B9_%D0%B1%D0%BB%D0%B8%D0%B7%D0%BE%D1%81%D1%82%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print(sys.executable)

# 1. Загрузка модели FRIDA
print("Загрузка модели FRIDA...")
model = SentenceTransformer('ai-forever/FRIDA')
print("Модель загружена.")

# Префикс FRIDA. Согласно карточке модели (ai-forever/FRIDA), для симметричных
# задач оценки сходства (STS, поиск парафразов) используется префикс
# "paraphrase: ". Без префикса вход подаётся не в том формате, на котором
# модель обучалась, и эмбеддинги получаются хуже. Сравнение аспектных
# терминов и мнений — симметричная STS-задача, поэтому используется
# "paraphrase". Модель также рекомендует CLS pooling — SentenceTransformer
# применяет его автоматически при загрузке FRIDA.
FRIDA_PROMPT_NAME = "paraphrase"
FRIDA_PROMPT_FALLBACK = "paraphrase: "


# 2. Кодирование текстов с кэшированием и батчингом
_embedding_cache: dict = {}


def encode_texts(texts: list) -> dict:
    """Кодирует список текстов в эмбеддинги одним батчем с применением
    префикса FRIDA. Результаты кэшируются, повторно не пересчитываются.
    Возвращает словарь text -> embedding."""
    to_encode = [t for t in set(texts)
                 if t and t not in _embedding_cache]
    if to_encode:
        try:
            # предпочтительный способ: prompt_name (новые версии S-T)
            embs = model.encode(
                to_encode,
                prompt_name=FRIDA_PROMPT_NAME,
                batch_size=32,
                show_progress_bar=False,
            )
        except (TypeError, ValueError):
            # запасной способ: ручной префикс (старые версии S-T)
            embs = model.encode(
                [FRIDA_PROMPT_FALLBACK + t for t in to_encode],
                batch_size=32,
                show_progress_bar=False,
            )
        for t, e in zip(to_encode, embs):
            _embedding_cache[t] = e
    return {t: _embedding_cache[t] for t in texts if t}


def compute_similarity(text1, text2) -> float:
    """Косинусное сходство двух текстов. Пустые/NaN -> 0.0."""
    if pd.isna(text1) or pd.isna(text2) or text1 == "" or text2 == "":
        return 0.0
    emb = encode_texts([str(text1), str(text2)])
    e1, e2 = emb.get(str(text1)), emb.get(str(text2))
    if e1 is None or e2 is None:
        return 0.0
    return float(cosine_similarity([e1], [e2])[0][0])


# 3. Основная функция сравнения датасетов
def semantic_evaluation(golden_df, pred_df):
    """Сравнивает golden и prediction датасеты по семантической близости
    аспектных терминов и мнений.

    Обрабатывает три типа квадруплетов:
        - Сопоставленные пары  — есть в обоих датасетах
        - Пропущенные (FN)     — есть в золотом, нет в предсказанном
        - Лишние (FP)          — есть в предсказанном, нет в золотом

    Сопоставление квадруплетов внутри каждого отзыва выполняется как
    оптимальное паросочетание двудольного графа (linear_sum_assignment),
    что согласуется с основным кодом метрик и исключает произвольность
    жадного сопоставления.

    Возвращает:
        - results_df : DataFrame с детальной информацией по каждому кваду
        - summary    : словарь со статистикой
    """
    golden_df = golden_df.copy()
    pred_df = pred_df.copy()
    golden_df['review_id'] = golden_df['review_id'].astype(str)
    pred_df['review_id'] = pred_df['review_id'].astype(str)

    golden_grouped = golden_df.groupby('review_id')
    pred_grouped = pred_df.groupby('review_id')

    all_review_ids = set(golden_grouped.groups) | set(pred_grouped.groups)
    print(f"Найдено {len(all_review_ids)} review_id для сравнения.")

    # предварительное кодирование всех терминов и мнений одним проходом
    all_texts = []
    for col in ['aspect_term', 'opinion']:
        for df in (golden_df, pred_df):
            all_texts += [str(x) for x in df[col].dropna() if str(x) != ""]
    print(f"Кодирование {len(set(all_texts))} уникальных фрагментов...")
    encode_texts(all_texts)

    results = []

    for rid in tqdm(all_review_ids, desc="Обработка отзывов"):
        gold_rows = (golden_grouped.get_group(rid)
                     if rid in golden_grouped.groups else pd.DataFrame())
        pred_rows = (pred_grouped.get_group(rid)
                     if rid in pred_grouped.groups else pd.DataFrame())

        def to_quads(rows):
            quads = []
            for _, row in rows.iterrows():
                quads.append({
                    'aspect_term': row['aspect_term'],
                    'opinion':     row['opinion'],
                    'aspect':      str(row['aspect']).strip().lower(),
                    'sentiment':   str(row['sentiment']).strip().lower(),
                })
            return quads

        gold_quads = to_quads(gold_rows)
        pred_quads = to_quads(pred_rows)

        n_gold, n_pred = len(gold_quads), len(pred_quads)

        # Оптимальное паросочетание по семантическому сходству
        # Матрица сходства: для пар, не совпавших по aspect+sentiment,
        # сходство = -inf, чтобы такие пары не выбирались.
        matched_gold, matched_pred = set(), set()

        if n_gold > 0 and n_pred > 0:
            sim_matrix = np.full((n_gold, n_pred), -1.0e9, dtype=float)
            cache_sim = {}  # (gi, ji) -> (term_sim, op_sim)

            for gi, gq in enumerate(gold_quads):
                for ji, pq in enumerate(pred_quads):
                    if (gq['aspect'] != pq['aspect']
                            or gq['sentiment'] != pq['sentiment']):
                        continue
                    term_sim = compute_similarity(gq['aspect_term'],
                                                  pq['aspect_term'])
                    op_sim = compute_similarity(gq['opinion'],
                                                pq['opinion'])
                    cache_sim[(gi, ji)] = (term_sim, op_sim)
                    sim_matrix[gi, ji] = (term_sim + op_sim) / 2

            row_ind, col_ind = linear_sum_assignment(-sim_matrix)

            for gi, ji in zip(row_ind, col_ind):
                # отбрасываем пары с запрещённым (-inf) сходством
                if sim_matrix[gi, ji] <= -1.0e8:
                    continue
                term_sim, op_sim = cache_sim[(gi, ji)]
                matched_gold.add(gi)
                matched_pred.add(ji)
                gq, pq = gold_quads[gi], pred_quads[ji]
                results.append({
                    'review_id':              rid,
                    'match_type':             'matched',
                    'gold_aspect_term':       gq['aspect_term'],
                    'pred_aspect_term':       pq['aspect_term'],
                    'aspect_term_similarity': term_sim,
                    'gold_opinion':           gq['opinion'],
                    'pred_opinion':           pq['opinion'],
                    'opinion_similarity':     op_sim,
                    'aspect':                 gq['aspect'],
                    'sentiment':              gq['sentiment'],
                    'avg_similarity':         (term_sim + op_sim) / 2,
                })

        # Пропущенные квады (FN): есть в золотом, нет в предсказанном
        for gi, gq in enumerate(gold_quads):
            if gi in matched_gold:
                continue
            results.append({
                'review_id':              rid,
                'match_type':             'missed (FN)',
                'gold_aspect_term':       gq['aspect_term'],
                'pred_aspect_term':       None,
                'aspect_term_similarity': 0.0,
                'gold_opinion':           gq['opinion'],
                'pred_opinion':           None,
                'opinion_similarity':     0.0,
                'aspect':                 gq['aspect'],
                'sentiment':              gq['sentiment'],
                'avg_similarity':         0.0,
            })

        # Лишние квады (FP): есть в предсказанном, нет в золотом
        for ji, pq in enumerate(pred_quads):
            if ji in matched_pred:
                continue
            results.append({
                'review_id':              rid,
                'match_type':             'extra (FP)',
                'gold_aspect_term':       None,
                'pred_aspect_term':       pq['aspect_term'],
                'aspect_term_similarity': 0.0,
                'gold_opinion':           None,
                'pred_opinion':           pq['opinion'],
                'opinion_similarity':     0.0,
                'aspect':                 pq['aspect'],
                'sentiment':              pq['sentiment'],
                'avg_similarity':         0.0,
            })

    results_df = pd.DataFrame(results)

    # Сводная статистика
    if not results_df.empty:
        matched = results_df[results_df['match_type'] == 'matched']
        summary = {
            'total_quads':   len(results_df),
            'matched_pairs': len(matched),
            'missed_fn':     int((results_df['match_type'] == 'missed (FN)').sum()),
            'extra_fp':      int((results_df['match_type'] == 'extra (FP)').sum()),

            # Метрики считаются только по сопоставленным парам
            'aspect_term_similarity_mean':   matched['aspect_term_similarity'].mean(),
            'opinion_similarity_mean':       matched['opinion_similarity'].mean(),
            'avg_similarity_mean':           matched['avg_similarity'].mean(),

            'aspect_term_similarity_median': matched['aspect_term_similarity'].median(),
            'opinion_similarity_median':     matched['opinion_similarity'].median(),
            'avg_similarity_median':         matched['avg_similarity'].median(),

            'aspect_term_similarity_std':    matched['aspect_term_similarity'].std(),
            'opinion_similarity_std':        matched['opinion_similarity'].std(),
            'avg_similarity_std':            matched['avg_similarity'].std(),

            'aspect_term_q25': matched['aspect_term_similarity'].quantile(0.25),
            'aspect_term_q75': matched['aspect_term_similarity'].quantile(0.75),
            'opinion_q25':     matched['opinion_similarity'].quantile(0.25),
            'opinion_q75':     matched['opinion_similarity'].quantile(0.75),
            'avg_q25':         matched['avg_similarity'].quantile(0.25),
            'avg_q75':         matched['avg_similarity'].quantile(0.75),
        }
    else:
        summary = {'error': 'No pairs found'}

    return results_df, summary


# 4. Запуск
if __name__ == "__main__":
    golden_df = pd.read_csv('golden_dataset.csv')
    pred_df = pd.read_csv('predictions_dataset_gpt.csv')

    results_df, summary = semantic_evaluation(golden_df, pred_df)

    print("\n" + "=" * 60)
    print("         СВОДНАЯ СТАТИСТИКА СЕМАНТИЧЕСКОЙ БЛИЗОСТИ")
    print("=" * 60)
    print(f"Всего квадруплетов обработано : {summary.get('total_quads', 0)}")
    print(f"  Сопоставленные пары        : {summary.get('matched_pairs', 0)}")
    print(f"  Пропущенные (FN)           : {summary.get('missed_fn', 0)}")
    print(f"  Лишние (FP)                : {summary.get('extra_fp', 0)}")

    print("\n--- СРЕДНИЕ ЗНАЧЕНИЯ (по сопоставленным парам) ---")
    print(f"Среднее сходство аспектных терминов : {summary.get('aspect_term_similarity_mean', 0):.4f}")
    print(f"Среднее сходство мнений             : {summary.get('opinion_similarity_mean', 0):.4f}")
    print(f"Среднее комбинированное сходство    : {summary.get('avg_similarity_mean', 0):.4f}")

    print("\n--- МЕДИАНЫ ---")
    print(f"Медиана аспектных терминов : {summary.get('aspect_term_similarity_median', 0):.4f}")
    print(f"Медиана мнений             : {summary.get('opinion_similarity_median', 0):.4f}")
    print(f"Медиана комбинированного   : {summary.get('avg_similarity_median', 0):.4f}")

    print("\n--- СТАНДАРТНЫЕ ОТКЛОНЕНИЯ ---")
    print(f"Стд. откл. аспектных терминов : {summary.get('aspect_term_similarity_std', 0):.4f}")
    print(f"Стд. откл. мнений             : {summary.get('opinion_similarity_std', 0):.4f}")
    print(f"Стд. откл. комбинированного   : {summary.get('avg_similarity_std', 0):.4f}")

    results_df.to_csv('semantic_similarity_results.csv',
                      index=False, encoding='utf-8-sig')
    print("\nДетальные результаты сохранены в 'semantic_similarity_results.csv'")
    print("Колонка 'match_type': matched | missed (FN) | extra (FP)")